# FX Dataset — Health Check & Exploratory Analysis

**How to use this notebook**
1. Click **"+ Add Data"** (top right) and attach the file your data pipeline produced — the `.csv` or `.parquet` you got from the pipeline's "Save Version" output.
2. **Run → Run All.**
3. Scroll down. Every section below is a chart, a table, or a printed check — nothing to configure unless you want to point it at a specific file (see the `DATA_PATH` line in the next cell).

This notebook does not train anything. It only looks at the dataset itself: what's in it, whether it's clean, whether the target is sane, and what a "do-nothing" model would score — so you have real numbers to show your boss before spending GPU time on training.

In [ ]:
# =============================================================================
# 1. SETUP — finds and loads your file automatically
# =============================================================================
import os, glob, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, acf
from statsmodels.graphics.tsaplots import plot_acf

sns.set_theme(style="whitegrid", font_scale=0.95)
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

# Point this at your file if auto-detect doesn't find it, e.g.
#   DATA_PATH = "/kaggle/input/my-dataset/dcc_garch_calander_gdelt_final.csv"
DATA_PATH = "auto"


def _looks_like_the_dataset(path):
    """Peek at the header only — don't load the whole file just to check it.
    A name-based guess (e.g. matching 'gdelt' in the filename) would wrongly
    match a raw input file like gdelt.csv if one is still attached alongside
    the pipeline's real output — so this checks the actual columns instead.
    """
    try:
        if path.endswith(".parquet"):
            import pyarrow.parquet as pq
            cols = set(pq.ParquetFile(path).schema.names)
        else:
            cols = set(pd.read_csv(path, nrows=1).columns.tolist())
    except Exception:
        return False
    # NOTE: 'target' is deliberately NOT checked here — the data pipeline's
    # own output never contains it (it's derived from zscore downstream, in
    # the training notebook). zscore/spread/group_id are the pipeline's own
    # signature columns and are enough to identify the real file.
    return {"zscore", "spread", "group_id"}.issubset(cols)


def _find_dataset():
    if DATA_PATH != "auto":
        return DATA_PATH
    candidates = []
    for ext in ("*.parquet", "*.csv"):
        candidates += glob.glob(f"/kaggle/input/**/{ext}", recursive=True)
    if not candidates:
        raise FileNotFoundError(
            "No .csv or .parquet found under /kaggle/input. "
            "Use '+ Add Data' to attach the file your data pipeline produced, "
            "then Run All again."
        )
    # identify the real pipeline output by its columns, not its filename —
    # a raw economic_data/gdelt input file left attached from the data
    # pipeline notebook must never be mistaken for the final dataset
    matches = [c for c in candidates if _looks_like_the_dataset(c)]
    if matches:
        matches.sort(key=os.path.getsize, reverse=True)
        return matches[0]
    # nothing has the expected columns — fall back to the biggest file and
    # let the shape/column printout below make it obvious if it's wrong
    candidates.sort(key=os.path.getsize, reverse=True)
    print("⚠️  could not confirm any file has zscore/target/group_id columns — "
          "picking the largest file. Check the shape/columns printed below; "
          "set DATA_PATH explicitly above if this is wrong.")
    return candidates[0]


path = _find_dataset()
print(f"Loading: {path}")
df = pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["group_id", "Date"]).reset_index(drop=True)

print(f"Shape:      {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Groups:     {df['group_id'].nunique()}")
print(f"Date range: {df['Date'].min().date()} -> {df['Date'].max().date()}")


In [ ]:
# =============================================================================
# `target` isn't in the pipeline's file — it's created downstream, in the
# training notebook, from `zscore` itself: tomorrow's change in the z-score.
# Built here with the SAME fix used in the training notebook: group BOTH the
# diff and the shift, so a group boundary can never leak the next group's
# first value into this group's last row.
# =============================================================================
df["target"] = (
    df.groupby("group_id")["zscore"].diff()
      .groupby(df["group_id"]).shift(-1)
)
print(f"target created: {df['target'].notna().sum():,} usable rows "
      f"({df['target'].isna().sum():,} NaN — the last day of each group, "
      f"where there's no 'tomorrow' yet. That's expected, not a bug.)")


In [ ]:
# =============================================================================
# 2. WHAT'S IN THIS FILE — column map and memory footprint
# =============================================================================
ID_COLS = ["Date", "group", "group_id", "time_idx", "leg1", "leg2", "leg3"]


def _cat(col):
    c = col.lower()
    if col in ID_COLS: return "id"
    if col == "target": return "target"
    if col in ("zscore", "spread"): return "cointegration"
    if c.startswith("rho_"): return "dcc_correlation"
    if c.startswith("sigma_"): return "garch_volatility"
    if c.startswith("macro_"): return "macro_calendar"
    if any(k in c for k in ("shock_events", "goldstein", "tone", "mentions", "articles")): return "gdelt_news"
    if c.startswith("regime"): return "regime"
    if c == "event_flag": return "event_flag"
    if c.startswith("stale_"): return "data_quality_flag"
    return "other"


col_cat = pd.Series({c: _cat(c) for c in df.columns})
overview = col_cat.value_counts().rename("n_columns").to_frame()
print(overview)

print(f"\nMemory footprint: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB in memory")
print(f"dtypes: {df.dtypes.value_counts().to_dict()}")

fig, ax = plt.subplots(figsize=(7, 4))
overview["n_columns"].sort_values().plot.barh(ax=ax, color="#4C72B0")
ax.set_title("Columns by category")
ax.set_xlabel("number of columns")
plt.tight_layout(); plt.show()


In [ ]:
# =============================================================================
# 3. DATA QUALITY — missing values and constant columns
# =============================================================================
# A healthy export from the pipeline should show ZERO of both. If anything
# shows up here, that column is either broken upstream or genuinely carries
# no information for this particular dataset.
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != "time_idx"]

na_counts = df[num_cols].isna().sum()
na_counts = na_counts[na_counts > 0].sort_values(ascending=False)
const_cols = [c for c in num_cols if c not in ID_COLS and df[c].nunique(dropna=True) <= 1]

print(f"Columns with any missing values: {len(na_counts)}")
if len(na_counts):
    print(na_counts.head(15))
else:
    print("none")

print(f"\nConstant columns (dead weight — carry no information): {len(const_cols)}")
print(const_cols if const_cols else "none — every numeric column varies")


In [ ]:
# =============================================================================
# 4. COVERAGE PER GROUP — is every group equally represented?
# =============================================================================
grp_summary = df.groupby("group_id").agg(
    rows=("Date", "size"), start=("Date", "min"), end=("Date", "max"),
).sort_values("rows", ascending=False)
print(grp_summary)

fig, ax = plt.subplots(figsize=(8, max(3, 0.35 * len(grp_summary))))
grp_summary["rows"].sort_values().plot.barh(ax=ax, color="#55A868")
ax.set_title("Rows per group")
ax.set_xlabel("rows")
plt.tight_layout(); plt.show()


In [ ]:
# =============================================================================
# 5. TARGET AND Z-SCORE — what is the model actually being asked to predict?
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df["target"].dropna(), kde=True, ax=axes[0], color="#C44E52")
axes[0].set_title(f"target — next-day change in z-score\n"
                   f"mean={df['target'].mean():.4f}  std={df['target'].std():.4f}")
sns.histplot(df["zscore"].dropna(), kde=True, ax=axes[1], color="#4C72B0")
axes[1].set_title(f"zscore — the cointegration signal\n"
                   f"mean={df['zscore'].mean():.4f}  std={df['zscore'].std():.4f}")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(10, max(3, 0.3 * df["group_id"].nunique())))
order = df.groupby("group_id")["zscore"].median().sort_values().index
sns.boxplot(data=df, y="group_id", x="zscore", order=order, ax=ax, color="#8172B2")
ax.set_title("zscore distribution by group — is any single group an outlier?")
plt.tight_layout(); plt.show()


In [ ]:
# =============================================================================
# 6. TIME SERIES — does the signal actually move, or is it flat/broken?
# =============================================================================
sample_groups = df["group_id"].drop_duplicates().head(4).tolist()
fig, axes = plt.subplots(len(sample_groups), 1, figsize=(11, 2.6 * len(sample_groups)), sharex=True)
if len(sample_groups) == 1:
    axes = [axes]
for ax, gid in zip(axes, sample_groups):
    g = df[df["group_id"] == gid]
    ax.plot(g["Date"], g["zscore"], label="zscore", lw=0.9)
    ax.plot(g["Date"], g["spread"], label="spread", lw=0.7, alpha=0.6)
    ax.axhline(0, color="grey", lw=0.6, ls="--")
    ax.set_title(gid, fontsize=9, loc="left")
    ax.legend(fontsize=7, loc="upper right")
plt.tight_layout(); plt.show()


In [ ]:
# =============================================================================
# 7. FEATURE CORRELATIONS — which features actually relate to the target?
# =============================================================================
feature_cols = [c for c in num_cols if c not in ID_COLS + ["target"]]
corr_with_target = df[feature_cols + ["target"]].corr()["target"].drop("target")
top20 = corr_with_target.abs().sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(7, 6))
ordered = corr_with_target[top20.index].sort_values()
colors = ["#C44E52" if v < 0 else "#4C72B0" for v in ordered]
ordered.plot.barh(ax=ax, color=colors)
ax.set_title("Top 20 features by |correlation| with target")
ax.set_xlabel("correlation")
plt.tight_layout(); plt.show()

print(top20.round(4))

top_feats = top20.index.tolist()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[top_feats].corr(), cmap="coolwarm", center=0, square=True,
            cbar_kws={"shrink": 0.7}, ax=ax)
ax.set_title("Correlation between the top 20 features (spot duplicated information)")
plt.tight_layout(); plt.show()


In [ ]:
# =============================================================================
# 8. QUICK LEAKAGE SANITY CHECK
# =============================================================================
# target is tomorrow's change in zscore. A same-day feature correlating very
# strongly with it deserves a second look — either it's genuinely predictive
# (rare, and worth telling your boss), or it was accidentally built using
# information from tomorrow.
LEAK_THRESHOLD = 0.30
suspicious = corr_with_target[corr_with_target.abs() > LEAK_THRESHOLD]
if len(suspicious):
    print(f"⚠️  {len(suspicious)} feature(s) correlate with target above {LEAK_THRESHOLD}:")
    print(suspicious.sort_values(key=abs, ascending=False))
else:
    print(f"✅ no feature correlates with target above {LEAK_THRESHOLD} — no obvious "
          f"leakage signature. (Real predictive correlations in daily FX changes "
          f"are usually well under 0.1 — that's normal, not a weak dataset.)")


In [ ]:
# =============================================================================
# 9. IS THE TARGET PREDICTABLE AT ALL? (autocorrelation + stationarity)
# =============================================================================
g0id = df["group_id"].iloc[0]
g0 = df[df["group_id"] == g0id].sort_values("Date")

adf_stat, adf_p, *_ = adfuller(g0["zscore"].dropna())
print(f"ADF test on zscore ({g0id}): stat={adf_stat:.3f}  p-value={adf_p:.4f}  "
      f"({'stationary — good' if adf_p < 0.05 else 'NOT stationary — worth checking the VECM fit'})")

fig, ax = plt.subplots(figsize=(9, 3.5))
plot_acf(g0["target"].dropna(), lags=20, ax=ax, title=f"Autocorrelation of target — {g0id}")
plt.tight_layout(); plt.show()

target_acf1 = acf(df["target"].dropna(), nlags=1)[1]
print(f"\nLag-1 autocorrelation of target (all groups pooled): {target_acf1:.4f}")
print("Close to 0 is NORMAL for a daily z-score change — tomorrow's move isn't "
      "just a copy of today's. It does not mean the target is unpredictable "
      "from the other features, only from its own recent past.")


In [ ]:
# =============================================================================
# 10. REGIMES — how much time does the market spend in each state?
# =============================================================================
regime_cols = [c for c in df.columns if c.startswith("regime")]
fig, axes = plt.subplots(1, len(regime_cols), figsize=(4.2 * len(regime_cols), 3.5))
if len(regime_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, regime_cols):
    df[col].value_counts(normalize=True).sort_index().plot.bar(ax=ax, color="#4C72B0")
    ax.set_title(col)
    ax.set_ylabel("share of days")
plt.tight_layout(); plt.show()

# one group's regime timeline — confirms it actually switches, not just once
g0id = df["group_id"].iloc[0]
g0 = df[df["group_id"] == g0id]
fig, ax = plt.subplots(figsize=(11, 2.2))
sc = ax.scatter(g0["Date"], g0["regime"], c=g0["regime"], cmap="viridis", s=4)
ax.set_yticks(sorted(g0["regime"].dropna().unique()))
ax.set_title(f"regime over time — {g0id}")
plt.tight_layout(); plt.show()


In [ ]:
# =============================================================================
# 11. EVENT_FLAG — how often does a news/macro shock day fire?
# =============================================================================
rate = df["event_flag"].mean()
print(f"Overall event_flag rate: {rate:.1%}  "
      f"({'looks healthy' if 0.10 < rate < 0.35 else 'check this — expected roughly 15-30%'})")

by_group = df.groupby("group_id")["event_flag"].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, max(3, 0.3 * len(by_group))))
by_group.plot.barh(ax=ax, color="#C44E52")
ax.set_title("event_flag rate by group")
ax.set_xlabel("share of days flagged")
plt.tight_layout(); plt.show()

monthly = df.set_index("Date")["event_flag"].resample("ME").mean()
fig, ax = plt.subplots(figsize=(11, 3))
monthly.plot(ax=ax, color="#C44E52")
ax.set_title("event_flag rate over time (monthly average, all groups)")
ax.set_ylabel("share of days flagged")
plt.tight_layout(); plt.show()


In [ ]:
# =============================================================================
# 12. VOLATILITY AND CORRELATION OVER TIME
# =============================================================================
sigma_cols = [c for c in df.columns if c.startswith("sigma_")]
rho_cols = [c for c in df.columns if c.startswith("rho_") and "change" not in c]

g0id = df["group_id"].iloc[0]
g0 = df[df["group_id"] == g0id].set_index("Date")

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
g0[sigma_cols].plot(ax=axes[0], lw=0.8)
axes[0].set_title(f"GARCH volatility (sigma) — {g0id}")
g0[rho_cols].plot(ax=axes[1], lw=0.8)
axes[1].axhline(0, color="grey", lw=0.6, ls="--")
axes[1].set_title(f"DCC correlation (rho) — {g0id}")
plt.tight_layout(); plt.show()


In [ ]:
# =============================================================================
# 13. GDELT NEWS FEATURES
# =============================================================================
gdelt_cols = [c for c in df.columns
              if c in num_cols and any(
                  k in c.lower() for k in
                  ("shock_events", "goldstein", "tone", "mentions", "articles"))][:8]

n = len(gdelt_cols)
if n:
    rows = (n + 1) // 2
    fig, axes = plt.subplots(rows, 2, figsize=(11, 2.6 * rows))
    axes = axes.flatten()
    for ax, col in zip(axes, gdelt_cols):
        sns.histplot(df[col].dropna(), ax=ax, color="#55A868", bins=40)
        ax.set_title(col, fontsize=9)
    for ax in axes[n:]:
        ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("No GDELT columns found in this file.")


In [ ]:
# =============================================================================
# 14. BASELINE SANITY CHECK — the floor any trained model has to beat
# =============================================================================
# Before trusting any model's MAE, compare it to these three "dumb" baselines.
# If a model can't beat "predict zero", it isn't learning anything — it's
# just riding the fact that most daily moves are small.
y = df["target"].dropna()
mae_zero = y.abs().mean()

grp_mean = df.groupby("group_id")["target"].transform("mean")
mae_group_mean = (df["target"] - grp_mean).abs().mean()

naive = df.groupby("group_id")["target"].shift(1)
mae_naive = (df["target"] - naive).dropna().abs().mean()

baselines = pd.Series({
    "predict zero": mae_zero,
    "predict group mean": mae_group_mean,
    "predict yesterday's move (naive persistence)": mae_naive,
}).sort_values()

fig, ax = plt.subplots(figsize=(7, 3))
baselines.plot.barh(ax=ax, color="#4C72B0")
ax.set_title("Baseline MAE — any trained model should score BELOW these")
ax.set_xlabel("MAE")
plt.tight_layout(); plt.show()
print(baselines.round(5))


In [ ]:
# =============================================================================
# 15. SUMMARY — the numbers to put in front of your boss
# =============================================================================
print("=" * 70)
print("DATASET HEALTH SUMMARY")
print("=" * 70)
print(f"Rows:                 {len(df):,}")
print(f"Columns:              {df.shape[1]}")
print(f"Groups:               {df['group_id'].nunique()}")
print(f"Date range:           {df['Date'].min().date()} -> {df['Date'].max().date()}")
print(f"Missing values:       {int(df[num_cols].isna().sum().sum()):,}")
print(f"Constant columns:     {len(const_cols)}")
print(f"event_flag rate:      {df['event_flag'].mean():.1%}")
print(f"Best baseline MAE:    {baselines.min():.5f}  ({baselines.idxmin()})")
print(f"Suspicious features:  {len(suspicious)}  (|corr| with target > {LEAK_THRESHOLD})")
print("=" * 70)
print("A trained model is only worth deploying if its test MAE beats the")
print("best baseline above by a meaningful margin, not just a rounding error.")
